# SVM and k-Nearest Neighbours

## Overview

**SVM** finds the maximum-margin hyperplane separating classes. The kernel trick maps data into higher-dimensional space to handle non-linear boundaries. Excellent for high-dimensional data; requires feature scaling.

**k-NN** classifies by majority vote among the k nearest training points. Non-parametric, no training phase, but slow at prediction and sensitive to irrelevant features.

| Model | Strengths | Weaknesses |
|---|---|---|
| SVM (RBF) | High-dimensional, robust to outliers | Slow for large n, needs scaling |
| SVM (linear) | Very high-dimensional (text) | May underfit complex boundaries |
| k-NN | Intuitive, no assumptions | Slow prediction, curse of dimensionality |

Both require feature standardisation.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
from scipy.special import expit

rng = np.random.default_rng(42)
n = 400
elevation  = rng.uniform(50, 400, n)
nitrate    = rng.gamma(2, 2, n)
phosphorus = rng.gamma(1.5, 1.5, n)
ph         = rng.normal(7.2, 0.5, n)
log_odds   = -2 + 0.004*elevation - 0.2*nitrate + 0.35*ph
label = (expit(log_odds) > 0.5).astype(int)
X = np.column_stack([elevation, nitrate, phosphorus, ph])
feat_names = ["elevation","nitrate","phosphorus","ph"]
X_tr, X_te, y_tr, y_te = train_test_split(X, label, test_size=0.25,
                                            stratify=label, random_state=42)

---
## SVM with RBF Kernel (Pipeline with Scaling)

In [ ]:
svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svm",    SVC(kernel="rbf", C=1.0, gamma="scale",
                   probability=True, random_state=42))
])
svm_pipe.fit(X_tr, y_tr)
print(f"SVM Test accuracy: {svm_pipe.score(X_te, y_te):.3f}")
print(f"SVM Test AUC:      {roc_auc_score(y_te, svm_pipe.predict_proba(X_te)[:,1]):.3f}")
print(classification_report(y_te, svm_pipe.predict(X_te), target_names=["absent","present"]))

---
## SVM Hyperparameter Tuning (C and gamma)

In [ ]:
param_grid = {"svm__C": [0.1, 1, 10, 100], "svm__gamma": [0.001, 0.01, 0.1, "scale"]}
grid = GridSearchCV(svm_pipe, param_grid, cv=5, scoring="roc_auc", n_jobs=-1)
grid.fit(X_tr, y_tr)
print(f"Best params: {grid.best_params_}")
print(f"Best CV AUC: {grid.best_score_:.3f}")
print(f"Test AUC:    {roc_auc_score(y_te, grid.predict_proba(X_te)[:,1]):.3f}")

---
## k-NN: Choosing k

In [ ]:
ks = range(1, 31)
knn_pipe = Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsClassifier())])
cv_scores = []
for k in ks:
    knn_pipe.set_params(knn__n_neighbors=k)
    scores = cross_val_score(knn_pipe, X_tr, y_tr, cv=5, scoring="roc_auc")
    cv_scores.append(scores.mean())
best_k = ks[np.argmax(cv_scores)]
plt.figure(figsize=(8,4))
plt.plot(ks, cv_scores, "o-", color="steelblue", lw=1.5)
plt.axvline(best_k, color="#e74c3c", lw=1.5, linestyle="--", label=f"Best k={best_k}")
plt.xlabel("k"); plt.ylabel("CV AUC-ROC")
plt.title("k-NN: Cross-Validated AUC by k")
plt.legend(); plt.tight_layout(); plt.show()
knn_best = Pipeline([("scaler", StandardScaler()),("knn", KNeighborsClassifier(n_neighbors=best_k))])
knn_best.fit(X_tr, y_tr)
print(f"k={best_k} Test AUC: {roc_auc_score(y_te, knn_best.predict_proba(X_te)[:,1]):.3f}")

---
## ROC Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
for model, name, color in [
    (grid, "SVM (tuned)", "#e74c3c"),
    (knn_best, f"k-NN (k={best_k})", "steelblue")
]:
    RocCurveDisplay.from_estimator(model, X_te, y_te, ax=ax, name=name, color=color)
ax.set_title("ROC Curve Comparison")
plt.tight_layout(); plt.show()

---

## Common Pitfalls

**1. Fitting SVM without scaling features**  
SVM optimises a margin in feature space. Features on large scales dominate the distance calculation. Always include `StandardScaler` in a pipeline before SVM -- never scale before the split.

**2. Using k=1 for k-NN (or any odd-small k)**  
k=1 memorises training data and is maximally sensitive to noise. Small k gives low bias but high variance. Choose k via cross-validation; k=sqrt(n_train) is a common starting point.

**3. Using SVM with `probability=True` without understanding its cost**  
Probability calibration in sklearn's SVM uses Platt scaling (an internal cross-validation), which significantly increases training time. Only set `probability=True` when you need predicted probabilities.

**4. Treating k-NN as scalable to large datasets**  
k-NN stores all training points and computes distances at prediction time. Prediction cost scales as O(n * p). For n > 10,000 use approximate nearest-neighbours (e.g. `sklearn.neighbors.BallTree` with `algorithm="ball_tree"`) or switch to a parametric model.

**5. Searching SVM hyperparameters on the test set**  
C and gamma must be selected by cross-validation on the training set only. Evaluating multiple hyperparameter combinations on the test set inflates reported AUC. Always use `GridSearchCV` on training data and report a single final test evaluation.
---
*python_methods_library - Samantha McGarrigle*